# 05_1. QA 실행과 embedding study — Pegasus

이 notebook 은 **Pegasus 만** 대상으로 한다.
다른 topology 는 짝 notebook(`05_2_run_qa_zephyr.ipynb`)에서 실행한다.

Advantage 세대. P16 기준 5640 노드, 큐빗당 연결도 15.

두 notebook 을 나눈 이유:

- 한 notebook 에서 두 topology 를 연속 실행하면 앞쪽이 끝나야 뒤쪽이 시작된다.
  Pegasus 20개가 수십 시간 걸리면 Zephyr 는 손도 못 댄다.
- 결과 merge key 에 `graph_id` 와 `topology_label` 이 들어 있어
  **따로 실행해도 같은 파일에 안전하게 합쳐진다.** 덮어쓰지 않는다.
- 단, topology 비교가 목적이면 `embedding.timeout`, `tries`, `seeds` 등
  **탐색 예산을 양쪽 동일하게** 유지해야 한다. 다르면 비교가 성립하지 않는다.

확인 항목:
1. 접근 가능한 QPU 목록 출력
2. topology 및 graph identity 검증
3. controlled trajectory embedding
4. end-to-end QA Benders
5. 성공한 embedding seed **전부** QA 실행
6. chain strength 출력
7. chain-break fraction 기록
8. ideal graph 와 actual working graph 결과 **분리**

**QPU 접근이 불가능해도 이 notebook 은 끝까지 실행된다.**
접근 실패는 `NO_QPU_ACCESS` 로 기록하고, ideal embedding study 는 계속 진행한다.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from cflp_bd_qa_ex2.config.loader import load_config
from cflp_bd_qa_ex2.data.io import load_all_instances

cfg = load_config(ROOT / "config" / "experiments" / "pilot.yaml")
instances = load_all_instances(ROOT / "data" / "raw")
by_id = {i.instance_id: i for i in instances}
# 이 notebook 이 담당하는 topology. 짝 notebook 과 이 값만 다르다.
TARGET_TOPOLOGY = "pegasus"

print("configuration_hash:", cfg["configuration_hash"][:16])
print("TARGET_TOPOLOGY   :", TARGET_TOPOLOGY)
print("qa.profile        :", cfg["qa"]["profile"])


## 1. QPU preflight

정적으로 단정하지 않고 실행 시점에 확인한다.
토큰은 출력하지 않는다.

In [ ]:
from cflp_bd_qa_ex2.embedding.targets import list_accessible_qpus, select_actual_qpu
from cflp_bd_qa_ex2.status import Status

status, solvers, message = list_accessible_qpus(cfg["qa"]["profile"])
print("status :", status)
print("message:", message)

if solvers:
    display(pd.DataFrame(solvers))
else:
    print("접근 가능한 QPU 가 없다. ideal target 만 사용한다.")


## 2. Topology 와 graph identity 검증

선택 규칙(명세 14절):

1. `solver_name` 이 있으면 해당 solver 사용
2. topology/shape 가 다르면 `TOPOLOGY_MISMATCH`
3. 없으면 목록 조회 -> 1개면 자동 선택 / 0개면 `NO_QPU_ACCESS` / 2개 이상이면 `AMBIGUOUS_QPU_SELECTION`

`target_graph_fingerprint` 는 노드/엣지 집합 전체의 해시이므로 **같은 solver 이름이라도
결함이 다르면 지문이 달라진다.**

In [ ]:
# 이 notebook 은 TARGET_TOPOLOGY 하나만 선택한다.
actual_targets = []
selection_rows = []
for key in [TARGET_TOPOLOGY]:
    selection = select_actual_qpu(key, cfg)
    selection_rows.append({
        "topology_key": key,
        "status": str(selection.status),
        "message": selection.message,
        "available_solvers": ", ".join(selection.available_solvers),
    })
    if selection.status is Status.SUCCESS and selection.target is not None:
        actual_targets.append(selection.target)

pd.DataFrame(selection_rows)


In [ ]:
if actual_targets:
    display(pd.DataFrame([t.as_record() for t in actual_targets]))
else:
    print("actual working graph 없음 -> QA sampling 은 수행하지 않는다.")


## 3. Ideal target graph

이상적 P16 / Z12 는 결함이 없는 완전한 그래프이다.
**여기서는 QA sampling 을 수행하지 않는다.** embedding 관측만 한다.

In [ ]:
from cflp_bd_qa_ex2.embedding.targets import build_ideal_targets

# ideal target 도 이 notebook 의 topology 만 남긴다.
# 두 notebook 이 같은 ideal target 을 중복 임베딩하면 시간만 쓰고
# mapping 파일에 __dupN 이 생긴다.
ideal_targets = [
    t for t in build_ideal_targets(cfg) if t.topology == TARGET_TOPOLOGY
]
pd.DataFrame([t.as_record() for t in ideal_targets])[
    ["topology_label", "topology", "topology_shape", "target_mode",
     "target_nodes", "target_edges"]
]


## 4. Controlled trajectory embedding

Gurobi-Benders 가 만든 동일한 cut trajectory 의 모든 iteration snapshot 을
각 target 에 임베딩한다. 각 snapshot 에 embedding seed 를 **모두** 적용한다.

`minorminer` 실패는 비임베딩성을 증명하지 않는다.
"주어진 search budget 에서 embedding 을 찾지 못함" 으로 기록한다.

In [ ]:
from cflp_bd_qa_ex2.benders.cuts import build_optimality_cut
from cflp_bd_qa_ex2.benders.solver import GurobiMasterProvider, run_benders
from cflp_bd_qa_ex2.embedding.embedder import run_embedding_trials
from cflp_bd_qa_ex2.embedding.storage import embedding_filename, save_embedding
from cflp_bd_qa_ex2.qubo.builder import build_master_qubo
from cflp_bd_qa_ex2.qubo.diagnostics import logical_graph, qubo_metrics, source_graph_fingerprint
from cflp_bd_qa_ex2.evaluation.results import save_embedding_checkpoint
from cflp_bd_qa_ex2.status import TrajectorySource

targets = ideal_targets + actual_targets

# ===== 실행 대상 지정 =====
# **두 topology notebook 의 값이 같아야 비교가 성립한다.**
# 한쪽만 줄이면 topology 효과와 instance 효과가 섞인다.
# 기본값은 evaluation/target_selection.py 에 공통으로 두고, 여기서는 그것을 쓴다.
from cflp_bd_qa_ex2.evaluation.target_selection import (
    DEFAULT_CONTROLLED_SEEDS, DEFAULT_CONTROLLED_SIZES,
    DEFAULT_E2E_SEEDS, DEFAULT_E2E_SIZES,
    compare_with_other_topology, describe_selection, pick_instances,
)

# 바꾸려면 **양쪽 notebook 을 같이** 바꾼다.
# sizes 와 seeds 는 AND 로 적용된다. 예를 들어 sizes=["4x12","8x25"], seeds=[100] 은
# 8x25 를 선택하지 않는다(8x25 의 seed 는 200~204).
CONTROLLED_SIZES = DEFAULT_CONTROLLED_SIZES
CONTROLLED_SEEDS = DEFAULT_CONTROLLED_SEEDS
E2E_SIZES = DEFAULT_E2E_SIZES
E2E_SEEDS = DEFAULT_E2E_SEEDS

controlled_info = describe_selection(instances, CONTROLLED_SIZES, CONTROLLED_SEEDS, "controlled")
e2e_info = describe_selection(instances, E2E_SIZES, E2E_SEEDS, "end-to-end")
target_ids = controlled_info["instance_ids"]
e2e_instance_ids = e2e_info["instance_ids"]

print(f"사용 가능한 크기: {sorted({i.name for i in instances})}")
for info in (controlled_info, e2e_info):
    print(f"{info['label']:11s} {info['num_selected']:2d}개: {info['instance_ids']}")
    for w in info["warnings"]:
        print(f"  경고: {w}")

# 다른 topology 가 이미 돌린 대상과 같은지 확인한다.
consistency = compare_with_other_topology(
    ROOT / "results" / "qa", TARGET_TOPOLOGY, e2e_instance_ids, cfg["configuration_hash"]
)
print(f"\ntopology 간 대상 일치: {consistency['status']} - {consistency['message']}")
if consistency["status"] == "mismatch":
    print(f"  여기에만: {consistency['only_here']}")
    print(f"  거기에만: {consistency['only_there']}")
    print("  => 이대로 비교하면 topology 효과를 분리할 수 없다")

records = []
for tid in target_ids:
    inst = by_id[tid]
    trace = run_benders(inst, GurobiMasterProvider(), cfg)
    cuts, snapshots = [], [[]]
    # **실제로 MP 에 추가된 cut 만** 사용한다.
    # 수렴 iteration 에서 생성되었으나 활성화되지 않은 마지막 cut 을 포함하면
    # 존재한 적 없는 MP 상태를 측정하게 된다.
    for snapshot in trace.cut_snapshots:
        if not snapshot.get("cut_activated", False):
            continue
        cuts = cuts + [build_optimality_cut(
            snapshot["cut_index"], inst.capacity.astype(float),
            np.asarray(snapshot["cut_dual_u"], dtype=float),
            np.asarray(snapshot["cut_dual_v"], dtype=float),
            snapshot["benders_iteration"], snapshot["dual_minimality_status"],
        )]
        snapshots.append(list(cuts))

    for iteration, cut_set in enumerate(snapshots):
        qubo = build_master_qubo(inst, cut_set, cfg)
        source = logical_graph(qubo)
        fingerprint = source_graph_fingerprint(source)
        metrics = qubo_metrics(qubo)
        for target in targets:
            for trial in run_embedding_trials(source, target, cfg):
                record = {
                    "instance_id": tid, "instance_name": inst.name,
                    "instance_seed": inst.seed, "formulation": "ms_cflp_benders_mp",
                    "trajectory_source": str(TrajectorySource.GUROBI_CONTROLLED),
                    "benders_iteration": iteration, "num_benders_cuts": len(cut_set),
                    "timeout": int(cfg["embedding"]["timeout"]),
                    "tries": int(cfg["embedding"]["tries"]),
                    "source_graph_fingerprint": fingerprint,
                    **target.as_record(),
                    **{k: metrics[k] for k in ("logical_variables", "logical_edges", "logical_density")},
                    **qubo.constraint_metadata,
                    **trial.as_record(),
                }
                records.append(record)
                if trial.success and trial.embedding is not None:
                    # 저장 경로를 반드시 CSV 에 남긴다. __dupN 이 생겨도 추적 가능해야 한다.
                    record["embedding_file"] = str(save_embedding(
                        trial.embedding, ROOT / "results" / "embedding" / "mappings", record,
                        embedding_filename(
                            tid, iteration, target.label, target.solver_id, target.graph_id,
                            trial.embedding_seed, int(cfg["embedding"]["timeout"]),
                            int(cfg["embedding"]["tries"]), fingerprint,
                        ),
                    ))
                else:
                    record["embedding_file"] = None

    # 인스턴스 하나가 끝날 때마다 저장한다.
    # 마지막에 한꺼번에 저장하면 16x50 / 25x50 도중 중단 시
    # 앞선 인스턴스 결과까지 소실된다.
    saved = save_embedding_checkpoint(ROOT / "results", cfg, records)
    print(f"  [{tid}] controlled embedding checkpoint: {saved}행", flush=True)

embedding_frame = pd.DataFrame(records)
print(f"embedding trial {len(embedding_frame)}행")
embedding_frame.groupby(["topology_label", "target_mode"])["success"].agg(["mean", "count"])


### 실패 행 보존 확인

성공한 seed 만 남기지 않는다. 실패 행의 chain metric 은 `NA` 이다.

In [ ]:
print("embedding_status 분포:")
print(embedding_frame["embedding_status"].value_counts())
failed = embedding_frame[~embedding_frame["success"].astype(bool)]
print(f"\n실패 행 {len(failed)}개 보존")
if len(failed):
    display(failed[["instance_id", "benders_iteration", "topology_label",
                    "embedding_seed", "embedding_status", "physical_qubits"]].head())


## 5. 성공한 embedding seed 전부 QA 실행

**가장 짧은 chain 이나 가장 적은 physical qubit 의 embedding 만 고르지 않는다.**
성공한 seed 를 전부 QA run 으로 연결한다.

이상적 target graph 에서는 QA 를 수행하지 않으므로, `actual_targets` 가 비어 있으면
이 절은 건너뛴다.

In [ ]:
from cflp_bd_qa_ex2.solvers.qa import compute_chain_strength, run_qa

qa_rows = []
if actual_targets:
    # **접근 가능한 모든 actual target** 에 대해 실행한다 (Pegasus / Zephyr 모두).
    for target in actual_targets:
        inst = by_id[target_ids[0]]
        qubo = build_master_qubo(inst, [], cfg)
        source = logical_graph(qubo)
        for trial in run_embedding_trials(source, target, cfg):
            if not (trial.success and trial.embedding):
                continue
            output = run_qa(qubo, trial.embedding, target.solver_id, target.graph_id,
                            cfg, trial.embedding_seed, qa_repeat_id=0)
            qa_rows.append({
                "topology_label": target.label,
                "solver_id": target.solver_id,
                **{k: v for k, v in output.items() if k != "samples"},
            })
    display(pd.DataFrame(qa_rows))
else:
    print("actual working graph 가 없어 QA sampling 을 건너뛴다 (NO_QPU_ACCESS).")


## 6. Chain strength

constraint `penalty_alpha` 와 **전혀 다른 개념**이다.

- `penalty_alpha` : 원래 문제의 제약 위반 비용. QUBO 를 만들 때 결정
- chain strength  : 같은 logical variable 의 physical qubit 묶음. **embedding 후** 결정

`uniform_torque_compensation(bqm, embedding, prefactor=1.414)` 로 계산한다.

In [ ]:
if actual_targets:
    print(pd.DataFrame(qa_rows)[
        ["embedding_seed", "chain_strength_method", "chain_strength_prefactor", "chain_strength"]
    ])
else:
    # ideal target 에서도 chain strength 값 자체는 계산해 볼 수 있다 (제출은 하지 않는다).
    inst = by_id[target_ids[0]]
    qubo = build_master_qubo(inst, [], cfg)
    trials = run_embedding_trials(logical_graph(qubo), ideal_targets[0], cfg)
    ok = [t for t in trials if t.success]
    if ok:
        print("참고: ideal embedding 기준 chain strength (QA 제출은 하지 않음)")
        print(compute_chain_strength(qubo, ok[0].embedding, cfg))


## 7. Chain-break fraction

`majority_vote` 로 고정한다. 다른 chain-break method 와 비교하지 않는다.

In [ ]:
if qa_rows:
    frame = pd.DataFrame(qa_rows)
    cols = [c for c in ("embedding_seed", "mean_chain_break_fraction",
                        "max_chain_break_fraction", "qpu_problem_id") if c in frame]
    display(frame[cols])
else:
    print("QA sample 이 없어 chain-break fraction 을 기록할 수 없다.")


## 8. End-to-end QA Benders

`sa_end_to_end` 와 마찬가지로, certified lower bound 는 QA 가 아니라
**매 iteration Gurobi continuous MP** 에서 온다.

In [ ]:
import time

from cflp_bd_qa_ex2.benders.solver import QuboMasterProvider
from cflp_bd_qa_ex2.logging_utils import setup_logging
from cflp_bd_qa_ex2.solvers.gurobi import probe_size_limits
from cflp_bd_qa_ex2.evaluation.results import save_qa_checkpoint
from cflp_bd_qa_ex2.solvers.qa_master import QAMasterProblemSolver

# sampler 알고리즘은 notebook 에 구현하지 않는다(명세 24절).
# scripts/run_qa.py 와 **같은 공용 클래스**를 호출한다.
# QAMasterProblemSolver(verbose=True) 가 iteration 별 진행 상황을 logger 로 출력한다.
logger = setup_logging()

# --- 실행 순서: 작은 인스턴스부터 ---
# load_all_instances 는 파일명 문자열 순이라 16x50 -> 25x50 -> 4x12 순이 된다.
# 가장 무거운 것부터 돌면 중간에 멈췄을 때 아무 결과도 남지 않는다.
ordered_ids = sorted(
    e2e_instance_ids,
    key=lambda tid: (by_id[tid].n_facilities * by_id[tid].n_customers, tid),
)

# --- SP 크기 선점검 ---
# SP 의 x_ij 는 n_facilities * n_customers 개다. 제한 라이선스를 넘으면
# QPU 를 쓰고 나서 SP 에서 실패하므로, **제출 전에** 걸러 QPU 시간을 낭비하지 않는다.
limits = probe_size_limits()
linear_limit = limits.get("linear_limit")
skipped_rows = []
runnable_ids = []
for tid in ordered_ids:
    inst = by_id[tid]
    sp_variables = inst.n_facilities * inst.n_customers
    if linear_limit is not None and sp_variables > linear_limit:
        skipped_rows.append({
            "instance_id": tid,
            "sp_variables": sp_variables,
            "linear_limit": linear_limit,
            "status": str(Status.GUROBI_SIZE_LIMIT),
            "message": "SP 가 제한 라이선스를 초과한다. QPU 제출 전에 건너뛴다",
        })
    else:
        runnable_ids.append(tid)

print(f"실행 대상 {len(runnable_ids)}개 / 건너뜀 {len(skipped_rows)}개 (GUROBI_SIZE_LIMIT)")
print("순서:", runnable_ids)
if skipped_rows:
    display(pd.DataFrame(skipped_rows))

qa_run_rows, qa_iteration_rows = [], []
qa_per_embedding_rows, qa_embedding_rows = [], []

if actual_targets:
    total = len(actual_targets) * len(runnable_ids)
    done = 0
    quota_exhausted = False
    for target in actual_targets:
        if quota_exhausted:
            break
        for tid in runnable_ids:
            if quota_exhausted:
                break
            done += 1
            started = time.perf_counter()
            print(f"\n===== [{done}/{total}] {tid} on {target.label} =====", flush=True)

            solver = QAMasterProblemSolver(
                target=target,
                instance_id=tid,
                cfg=cfg,
                mapping_dir=str(ROOT / "results" / "embedding" / "qa_mappings"),
                qa_records=qa_per_embedding_rows,
                embedding_records=qa_embedding_rows,
                verbose=True,
            )
            provider = QuboMasterProvider(TrajectorySource.QA_END_TO_END, solver)
            trace = run_benders(by_id[tid], provider, cfg)

            for it in trace.iterations:
                record = it.as_record()
                record.update(target.as_record())
                qa_iteration_rows.append(record)
            run_record = trace.run.as_record()
            run_record.update(target.as_record())
            qa_run_rows.append(run_record)

            # 인스턴스 하나가 끝날 때마다 즉시 저장한다.
            # 마지막에 한꺼번에 저장하면 인터럽트나 커널 재시작 시 전부 소실된다.
            written = save_qa_checkpoint(
                ROOT / "results", TARGET_TOPOLOGY, cfg,
                qa_embedding_rows, qa_per_embedding_rows,
                qa_iteration_rows, qa_run_rows,
            )

            print(
                f"----- {tid} 완료: {run_record['termination_status']}, "
                f"iterations={run_record['benders_iterations']}, "
                f"QPU 호출={run_record.get('num_qpu_calls')}, "
                f"{time.perf_counter() - started:.1f}초 "
                f"| checkpoint 저장 {written}",
                flush=True,
            )

            # 할당 시간이 소진되면 남은 인스턴스를 돌려도 전부 실패한다.
            # 루프를 멈추고 **여기까지 모은 결과는 그대로 보존**한다.
            if run_record["termination_status"] == str(Status.QPU_QUOTA_EXHAUSTED):
                quota_exhausted = True
                print(
                    "\n!!! QPU 할당 시간이 소진되었다. 남은 인스턴스 실행을 중단한다.\n"
                    "    여기까지의 결과는 아래 저장 셀을 그대로 실행하면 보존된다.",
                    flush=True,
                )

    display(pd.DataFrame(qa_run_rows))
    display(pd.DataFrame(qa_per_embedding_rows))
else:
    print("QPU 접근 불가 -> end-to-end QA Benders 를 실행하지 않는다.")


## 9. Ideal 과 actual 결과 분리 저장

두 결과를 **절대 합치지 않는다.** `target_mode` 열로 구분된다.

In [ ]:
from cflp_bd_qa_ex2.evaluation.results import (
    attach_provenance, save_embedding_checkpoint, save_qa_checkpoint, save_records,
)

formats = cfg["output"]["formats"]

# QA 결과 파일 이름에 topology 를 넣는다.
# merge key 만으로도 행은 분리되지만, 파일을 나누면 한쪽 notebook 을 다시 돌려도
# 다른 topology 의 파일을 건드리지 않는다.
SUFFIX = f"_{TARGET_TOPOLOGY}"

# controlled embedding study 는 run_embedding.py 와 같은 파일을 쓴다.
# merge key 에 topology_label 과 graph_id 가 있어 안전하게 합쳐진다.
# controlled embedding 은 루프 안에서 이미 checkpoint 되었다.
# 같은 함수를 다시 부르면 provenance 가 보존된 채 병합된다.
save_embedding_checkpoint(ROOT / "results", cfg, records)

# 루프 안에서 이미 checkpoint 로 저장되지만, 마지막에 한 번 더 저장해
# 인터럽트 직후 이 셀만 돌려도 복구되게 한다. merge key 가 같아 중복되지 않는다.
written = save_qa_checkpoint(
    ROOT / "results", TARGET_TOPOLOGY, cfg,
    qa_embedding_rows, qa_per_embedding_rows, qa_iteration_rows, qa_run_rows,
)
print("저장 행 수:", written)

# QPU 선택 상태를 그대로 저장한다.
# 항상 NO_QPU_ACCESS 로 적으면 AMBIGUOUS_QPU_SELECTION / TOPOLOGY_MISMATCH /
# SOLVER_FAILED 를 구분할 수 없다.
save_records(
    attach_provenance([{"record_type": "qa_preflight", **row} for row in selection_rows], cfg),
    ROOT / "results" / "qa", f"qa_preflight{SUFFIX}", formats,
    merge_keys=("record_type", "topology_key", "configuration_hash"),
)
print("preflight 저장:", [(r["topology_key"], r["status"]) for r in selection_rows])

if not qa_iteration_rows:
    print("actual working graph 가 없어 end-to-end QA 결과는 비어 있다 "
          "(상태는 위 qa_preflight 에 저장됨)")

print(f"저장 완료 (suffix='{SUFFIX}')")
for name, rows in [("qa_embedding_trials", qa_embedding_rows),
                   ("qa_per_embedding", qa_per_embedding_rows),
                   ("qa_iterations", qa_iteration_rows),
                   ("qa_runs", qa_run_rows)]:
    print(f"  {name}{SUFFIX}: {len(rows)}행")

embedding_frame.groupby("target_mode").size()


## 해석

- `target_mode` 가 `ideal` 뿐이면 이 실행에서 QPU 접근이 없었다는 뜻이다.
  이때의 embedding 결과는 **결함 없는 그래프에 대한 상한**이며,
  실제 하드웨어 결과를 대체하지 않는다.
- Zephyr 는 노드 수가 Pegasus 보다 적지만 연결도가 높아, 조밀한 source graph 에서
  더 짧은 chain 을 만드는 경향이 있는지 확인한다.
- embedding 실패 행은 삭제하지 않는다. 실패는 "이 search budget 에서 못 찾음" 이며
  비임베딩성이 아니다. 또한 관측된 한 embedding 의 physical qubit 수는
  해당 문제의 하한이 아니다.
- `auto_scale` 은 균일 스케일링이므로 coefficient dynamic range(최대/최소 비율)를
  바꾸지 않는다.